## Xenium QC Report Module

### What this notebook does

Aggregates the per-sample Xenium `metrics_summary.csv` files for a run into a single QC report CSV, then renders a multi-page PDF of QC visualizations. There's two ways to load the per-sample files into this notebook:

- **Download cell (below):** pull `metrics_summary.csv` files for a list of samples directly from the Xenium GCS bucket.
- **Option 1 — by HISE UUID:** download files by UUID, aggregate, and upload the result back to HISE.
- **Option 2 — from a local directory:** aggregate `metrics_summary.csv` files already on disk (skip Option 1 if you use this).

### Before running, set the following

**Download cell (`google.cloud.storage`):**
- `BUCKET_NAME` — GCS bucket holding the metrics files (no `gs://` prefix).
- `SAMPLES` — comma-separated list of sample IDs to fetch.

**Config cell:**
- `EXP_ID` *(required)* — experiment ID; also used to build the QC output paths.
- `RUN_NAME` *(required)* — run name, used in the report title.
- `uuid_list` *(optional)* — list of metrics-summary file UUIDs; required only for the Option 1 HISE upload flow.
- `project_short_name` *(optional)* — HISE project store to upload the aggregated file to; required only for Option 1.

The remaining values (`all_qc_report_path`, `output_dir`, `title_description`, `pdf_file_name`) are derived from `EXP_ID`/`RUN_NAME` and do not need to be edited.

In [7]:
import pandas as pd
import os
import datetime
import logging
import hisepy
import matplotlib.pyplot as plt
import plotly.express as px
from matplotlib.backends.backend_pdf import PdfPages
from google.cloud import storage
from pathlib import Path

In [8]:
# ── Required ──────────────────────────────────────────────────────────────────
EXP_ID   = "EXP-02160-02161"
RUN_NAME = "061926"   # e.g. "051126" (MMDDYY)

# ── Optional (leave as None if not uploading to HISE) ─────────────────────────
uuid_list           = None   # e.g. ["967a55cb-...", "7660b4c5-..."]
project_short_name  = None   # e.g. "aifiDevTest"

# ── Derived (no need to edit) ─────────────────────────────────────────────────
BASE_DIR            = Path("/home/workspace/xenium/qc") / EXP_ID
all_qc_report_path  = BASE_DIR / "csvs"
output_dir          = BASE_DIR
title_description   = f"Xenium QC report for run name {EXP_ID}_{RUN_NAME}"
pdf_file_name       = f"Xenium_qc_report_{EXP_ID}.pdf"

In [9]:
BASE_DIR = "/home/workspace"
BUCKET_NAME = "temp-xenium-hise-transfer"

CSV_OUTPUT = Path(BASE_DIR) / f"xenium/qc/{EXP_ID}/csvs"
HTML_OUTPUT = Path(BASE_DIR) / f"xenium/qc/{EXP_ID}/htmls"
SLIDE_IDS = "0074428, 0074431"

slide_list = [s.strip() for s in SLIDE_IDS.split(",")]
CSV_OUTPUT.mkdir(parents=True, exist_ok=True)
HTML_OUTPUT.mkdir(parents=True, exist_ok=True)

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
names = [blob.name for blob in bucket.list_blobs()]

for slide_id in slide_list:
    print(f"Looking for slide {slide_id}...")

    matches = [
        name for name in names
        if slide_id in name and name.endswith("metrics_summary.csv")
    ]

    if not matches:
        print(f"  ERROR: No match found for {slide_id}")
        continue

    print(f"  Found {len(matches)} sample(s) for slide {slide_id}")

    for match in matches:
        # Extract the run folder (e.g. output-XETG00195__0082215__TSS10546-041__20260605__173502)
        folder = match.split("/")[0]
        parts = folder.split("__")

        # parts: [output-XETG00195, 0082215, TSS10546-041, 20260605, 173502]
        tss_id = parts[2] if len(parts) > 2 else "UNKNOWN"
        prefix = f"{slide_id}__{tss_id}"

        # Download metrics_summary.csv
        csv_dst = CSV_OUTPUT / f"{prefix}_metric_summary.csv"
        bucket.blob(match).download_to_filename(str(csv_dst))
        print(f"  Done -> {csv_dst}")

        # Download analysis_summary.html from the same run folder
        html_blob = f"{folder}/analysis_summary.html"
        if html_blob in names:
            html_dst = HTML_OUTPUT / f"{prefix}_analysis_summary.html"
            bucket.blob(html_blob).download_to_filename(str(html_dst))
            print(f"  Done -> {html_dst}")
        else:
            print(f"  WARNING: No analysis_summary.html found in {folder}")

/home/workspace/environment/minimal/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/workspace/environment/minimal/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Looking for slide 0074428...
  Found 2 sample(s) for slide 0074428
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/csvs/0074428__TSS12578-001_metric_summary.csv
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/htmls/0074428__TSS12578-001_analysis_summary.html
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/csvs/0074428__TSS12578-002_metric_summary.csv
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/htmls/0074428__TSS12578-002_analysis_summary.html
Looking for slide 0074431...
  Found 3 sample(s) for slide 0074431
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/csvs/0074431__TSS10546-052_metric_summary.csv
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/htmls/0074431__TSS10546-052_analysis_summary.html
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/csvs/0074431__TSS12579-001_metric_summary.csv
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/htmls/0074431__TSS12579-001_analysis_summary.html
  Done -> /home/workspace/xenium/qc/EXP-02160-02161/csvs/0074431__

### Option 1: Starting from metrics summary file UUIDs

In [ ]:
def set_logging(log_dir, log_file_name):
    logger = logging.getLogger(__name__)
    fhandler = logging.FileHandler(filename=os.path.join(log_dir, log_file_name), mode='a')
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    fhandler.setFormatter(formatter)
    logger.addHandler(fhandler)
    logger.setLevel(logging.DEBUG)
    return logger

In [ ]:
def download_files(logger, input_dir, uuid_list):
    #####################################################
    # This function downloads the files per uuids and   #
    # move the files to specified input folder          #
    #####################################################

    # Apply hisepy.reader.cache_files function to download the specified sample files #
    hisepy.reader.cache_files(uuid_list)

    # Move all the uuid folders from 'cache' folder to the 'input_dir'
    for each_uuid in uuid_list:
        logger.info("Copying file {0} from /home/jupyter/cache/ to {1}/......".format(each_uuid, input_dir))
        os.system('cp -r /home/jupyter/cache/{0} {1}/.'.format(each_uuid, input_dir))

In [ ]:
def create_aggregated_qc(logger, input_dir, output_dir, uuid_list):
    """Parse the per-sample QC report for each UUID and aggregate into one CSV."""
    run_name_list = []
    df_to_concat = []
    for each_uuid in uuid_list:
        per_qc_report_path = os.path.join(input_dir, each_uuid, 'metrics_summary.csv')

        if not os.path.exists(per_qc_report_path):
            logger.warning(f"Could not find QC report for uuid {each_uuid}, skipping it")
            continue
        if os.path.getsize(per_qc_report_path) == 0:
            logger.warning(f"QC report for uuid {each_uuid} is empty, skipping it")
            continue

        logger.info(f"Found QC report for uuid {each_uuid}")
        df_per_sample_qc = pd.read_csv(per_qc_report_path)
        df_to_concat.append(df_per_sample_qc)
        run_name_list.append(df_per_sample_qc.iloc[0]['run_name'])

    if not df_to_concat:
        raise RuntimeError("No QC reports were successfully loaded")

    df_aggregated_qc_report = pd.concat(df_to_concat, ignore_index=True)

    # If every sample shares a run_name, use it as the output prefix
    if len(set(run_name_list)) == 1:
        output_qc_file_name = f"{run_name_list[0]}_xenium_qc_report.csv"
    else:
        output_qc_file_name = 'xenium_qc_report.csv'

    df_aggregated_qc_report.to_csv(os.path.join(output_dir, output_qc_file_name), index=False)
    return output_qc_file_name

In [ ]:
def upload_output_to_hise(project_short_name, uuid_list, title_description, output_qc_file_name, current_timestamp, output_dir):
	hisepy.upload_files(files=[os.path.join(output_dir, output_qc_file_name)], 
	                    project=project_short_name, 
	                    title=title_description,
	                    destination = '{0}/xenium_qc_report'.format(current_timestamp),
	                    input_file_ids=uuid_list)


In [ ]:
# Create a folder with timestamp in the current working directory #
current_timestamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
working_dir = os.path.join('/home/jupyter/execute_xenium_qc_module', current_timestamp)
if not os.path.exists(working_dir):
    os.makedirs(working_dir)

log_dir = os.path.join(working_dir, 'logs')
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

input_dir = os.path.join(working_dir, 'input')
if not os.path.exists(input_dir):
    os.makedirs(input_dir)

# Option 1 writes to its own timestamped working dir; keep it distinct from the
# config cell's `output_dir` (used by Option 2) so the two flows don't clobber each other.
o1_output_dir = os.path.join(working_dir, 'output')
if not os.path.exists(o1_output_dir):
    os.makedirs(o1_output_dir)

logger = set_logging(log_dir, 'xenium_qc_report.log')

# If uuid_list is not provided by user, print following warning #
if uuid_list and project_short_name and title_description:
    download_files(logger, input_dir, uuid_list)
    output_qc_file_name = create_aggregated_qc(logger, input_dir, o1_output_dir, uuid_list)
    title_description_with_timestamp = '{0}-{1}'.format(title_description, current_timestamp) 
    upload_output_to_hise(project_short_name, uuid_list, title_description_with_timestamp, output_qc_file_name, current_timestamp, o1_output_dir)
else:
    if not uuid_list:
        print("Warning: please provide uuid_list")
    if not project_short_name:
        print("Warning: please provide project_short_name")
    if not title_description:
        print("Warning: please provide title_description")

### Option 2: Starting from metrics summary file's location instead of UUIDs 
#### (please skip if Option 1 (UUID method) has been run)

In [10]:
# Derive the sample ID list from the metrics CSVs already in the csvs folder.
# Filenames look like "{slide_id}__{tss_id}_metric_summary.csv"; the sample ID
# is the full "{slide_id}__{tss_id}" prefix (the TSS id alone can repeat across
# slides, so it isn't unique on its own).
SUFFIX = "_metric_summary.csv"
sample_list = sorted(
    p.name[: -len(SUFFIX)]
    for p in Path(all_qc_report_path).glob(f"*{SUFFIX}")
)
print(f"Found {len(sample_list)} sample(s):")
for s in sample_list:
    print(f"  {s}")

Found 5 sample(s):
  0074428__TSS12578-001
  0074428__TSS12578-002
  0074431__TSS10546-052
  0074431__TSS12579-001
  0074431__TSS12579-002


In [12]:
# Aggregate exactly one metrics file per expected sample rather than globbing
# every CSV in the directory, so stale files from previous runs (or other
# experiments) can't be folded into the report.
csv_paths = []
for sample_id in sample_list:
    matches = sorted(Path(all_qc_report_path).glob(f"*{sample_id}*.csv"))
    if not matches:
        raise FileNotFoundError(f"No metrics summary CSV found for sample {sample_id} in {all_qc_report_path}")
    if len(matches) > 1:
        print(f"WARNING: multiple files for {sample_id}, using first: {matches[0].name}")
    csv_paths.append(matches[0])

print(f"Aggregating {len(csv_paths)} files:")
for p in csv_paths:
    print(f"  {p.name}")

df_aggregated_qc_report = pd.concat(
    [pd.read_csv(p) for p in csv_paths],
    ignore_index=True,
)
run_name_list = df_aggregated_qc_report['run_name'].tolist()

output_qc_file_name = 'xenium_qc_report.csv'
df_aggregated_qc_report.to_csv(Path(output_dir) / output_qc_file_name, index=False)

Aggregating 5 files:
  0074428__TSS12578-001_metric_summary.csv
  0074428__TSS12578-002_metric_summary.csv
  0074431__TSS10546-052_metric_summary.csv
  0074431__TSS12579-001_metric_summary.csv
  0074431__TSS12579-002_metric_summary.csv


### Setting up data for visualizations

In [13]:
# Drop all-NaN columns, then sanity-check the aggregated table before plotting.
plot_df = df_aggregated_qc_report.dropna(axis=1, how='all')

print(f"Aggregated {plot_df.shape[0]} samples x {plot_df.shape[1]} metrics")
print(f"Run name(s): {sorted(plot_df['run_name'].unique())}")
print(f"Columns: {list(plot_df.columns)}")

plot_df

Aggregated 5 samples x 30 metrics
Run name(s): ['EXP-02160-2161-061826']
Columns: ['run_name', 'cassette_name', 'region_name', 'panel_name', 'panel_design_id', 'region_area', 'total_cell_area', 'total_high_quality_decoded_transcripts', 'fraction_transcripts_decoded_q20', 'nuclear_transcripts_per_100um2', 'decoded_transcripts_per_100um2', 'adjusted_negative_control_probe_rate', 'adjusted_negative_control_codeword_rate', 'negative_control_probe_counts_per_control_per_cell', 'estimated_number_of_false_positive_transcripts_per_cell', 'num_cells_detected', 'fraction_empty_cells', 'cells_per_100um2', 'fraction_transcripts_assigned', 'median_genes_per_cell', 'median_transcripts_per_cell', 'thickness_transcripts_high_quality', 'stain_definition', 'segmented_cell_stain_frac', 'segmented_cell_boundary_frac', 'segmented_cell_interior_frac', 'segmented_cell_nuc_expansion_frac', 'segmented_cell_boundary_count', 'segmented_cell_interior_count', 'segmented_cell_nuc_expansion_count']


,run_name,cassette_name,region_name,panel_name,panel_design_id,region_area,total_cell_area,total_high_quality_decoded_transcripts,fraction_transcripts_decoded_q20,nuclear_transcripts_per_100um2,...,median_transcripts_per_cell,thickness_transcripts_high_quality,stain_definition,segmented_cell_stain_frac,segmented_cell_boundary_frac,segmented_cell_interior_frac,segmented_cell_nuc_expansion_frac,segmented_cell_boundary_count,segmented_cell_interior_count,segmented_cell_nuc_expansion_count
0,EXP-02160-2161-061826,EXP-02160,TSS12578-001,mMulti_480g,TV46KD,2.403353e+07,5.840209e+06,16981192,0.868758,278.761749,...,90,7.336843,xenium_cell_segmentation_stains_v1,0.967119,0.386470,0.580649,0.032881,38094,57234,3241
1,EXP-02160-2161-061826,EXP-02160,TSS12578-002,mMulti_480g,TV46KD,1.241988e+08,3.355476e+07,93313854,0.871292,285.649983,...,99,9.263619,xenium_cell_segmentation_stains_v1,0.965282,0.356133,0.609150,0.034718,188942,323177,18419
2,EXP-02160-2161-061826,EXP-02161,TSS10546-052,mMulti_480g,TV46KD,1.447482e+07,7.956828e+06,62576721,0.735057,801.061202,...,278,7.421273,xenium_cell_segmentation_stains_v1,0.993233,0.086554,0.906680,0.006767,17882,187320,1398
3,EXP-02160-2161-061826,EXP-02161,TSS12579-001,mMulti_480g,TV46KD,2.839141e+07,7.835628e+06,17797728,0.890008,205.202269,...,71,8.191776,xenium_cell_segmentation_stains_v1,0.950480,0.406745,0.543735,0.049520,51295,68571,6245
4,EXP-02160-2161-061826,EXP-02161,TSS12579-002,mMulti_480g,TV46KD,1.120606e+08,3.253268e+07,54307731,0.888397,168.577966,...,57,8.653043,xenium_cell_segmentation_stains_v1,0.957526,0.288492,0.669034,0.042474,155194,359906,22849


In [14]:
colors = ['#9e0142','#d53e4f','#f46d43','#fdae61','#fee08b','#e6f598','#abdda4','#66c2a5','#3288bd','#5e4fa2']
col = ['#d53e4f','#66c2a5','#5e4fa2']
pdf_pages = PdfPages(pdf_file_name)

In [15]:
def bar_plot(y_col, ylabel, title, color, df=None):
    df = plot_df if df is None else df
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df['region_name'], df[y_col], color=color)
    plt.setp(ax.get_xticklabels(), rotation=60, ha='right')
    ax.set_xlabel('Tissue ID')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    pdf_pages.savefig(fig)
    plt.close(fig)


def stacked_bar(value_cols, title, ylabel='Value', df=None, palette=None):
    df = plot_df if df is None else df
    palette = col if palette is None else palette
    fig, ax = plt.subplots(figsize=(9, 7))
    bottoms = pd.Series(0.0, index=df.index)
    for i, c in enumerate(value_cols):
        ax.bar(df['region_name'], df[c], bottom=bottoms, label=c, color=palette[i])
        bottoms = bottoms + df[c]
    ax.set_xlabel('Region Name')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, linestyle='--', color='gray', alpha=0.6)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    fig.tight_layout()
    pdf_pages.savefig(fig)
    plt.close(fig)

## Number of cells detected

In [16]:
bar_plot('num_cells_detected', 'Number of cells', 'Number of cells detected by region', colors[6])

## Transcripts and Genes per Tissue

In [17]:
plt.figure(figsize=(8, 5))
plt.bar(plot_df['region_name'], plot_df['median_transcripts_per_cell'], color=colors[7])
plt.plot(plot_df['region_name'], plot_df['median_genes_per_cell'], color=colors[1], marker='o', label='Median Genes per Cell')
plt.xticks(rotation=60, ha='right')
plt.xlabel('Tissue ID')
plt.ylabel('Median Transcripts per Cell')
plt.title('Median Transcripts per Cell by Region')
plt.legend()
plt.tight_layout()
pdf_pages.savefig()
plt.close()

## Estimated False Positive Transcripts

In [18]:
bar_plot(
    'estimated_number_of_false_positive_transcripts_per_cell',
    'Number of False Positive Transcripts per Cell',
    'Estimated False Positive Transcripts per Region',
    colors[1],
)

## Decoded Transcripts

In [19]:
plt.figure(figsize=(8, 5))
plt.scatter(plot_df['region_name'], plot_df['fraction_transcripts_decoded_q20'], color=colors[8], label='Fraction Transcripts Decoded Q20', marker='^', s=120)
plt.scatter(plot_df['region_name'], plot_df['fraction_transcripts_assigned'], color=colors[2], label='Fraction Transcripts Assigned', marker='o', s=100)
plt.scatter(plot_df['region_name'], plot_df['fraction_empty_cells'], color=colors[7], label='Fraction Empty cells', marker='x', s=100)
plt.xticks(rotation=60, ha='right')
plt.xlabel('Region Name')
plt.ylabel('Fraction')
plt.title('Decoded Transcript Fractions')
plt.ylim(0, 1)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, linestyle='--', color='gray', alpha=0.6)
plt.tight_layout()
pdf_pages.savefig()
plt.close()

## Cells per 100um sq

In [20]:
bar_plot('cells_per_100um2', 'N Cells', 'Cells per 100 um sq', colors[4])

## Segmentation Fractions and Counts

In [21]:
stacked_bar(
    [
        'segmented_cell_boundary_frac',
        'segmented_cell_interior_frac',
        'segmented_cell_nuc_expansion_frac',
    ],
    title='Segmentation Fractions by Tissue',
)

In [22]:
stacked_bar(
    [
        'segmented_cell_boundary_count',
        'segmented_cell_interior_count',
        'segmented_cell_nuc_expansion_count',
    ],
    title='Segmentation Counts by Tissue',
)

## Median Transcripts per panel

In [23]:
plt.figure(figsize=(8, 5))
panels = plot_df['panel_name'].unique()

plt.boxplot(
    [plot_df.loc[plot_df['panel_name'] == p, 'median_transcripts_per_cell'] for p in panels],
    labels=panels,
    showfliers=False,
)

for i, panel in enumerate(panels, start=1):
    panel_data = plot_df.loc[plot_df['panel_name'] == panel, 'median_transcripts_per_cell']
    plt.scatter(
        [i] * len(panel_data),
        panel_data,
        color=colors[9],
        alpha=0.8,
        s=50,
        label='Points' if i == 1 else "_",
    )

plt.xlabel('Panel Name')
plt.ylabel('Median Transcripts per Cell')
plt.title('Median Transcripts per Cell by Panel')
plt.tight_layout()
pdf_pages.savefig()
plt.close()

/tmp/ipykernel_807/1649344239.py:4: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(


In [24]:
plt.close()
pdf_pages.close()